# 🧪 Lab 01: The WholeStageCodegen Stage Diagnostics

**Mission Objective:** isolate `Range → Filter → Project` and identify the operators Spark fused into one generated JVM pipeline. This is a diagnostic lab, not a performance benchmark.

**Deterministic Guardrail:** one million generated rows, no shuffle, storage, UDF, or external data source.


### Step 1: Define the Spark diagnostic session
We create a local two-thread Spark session and print the runtime version plus the WholeStageCodegen setting.


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (SparkSession.builder.master("local[2]").appName("lab-01-wholestagecodegen-stage-diagnostics").config("spark.ui.enabled", "false").getOrCreate())
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)
print("WholeStageCodegen enabled:", spark.conf.get("spark.sql.codegen.wholeStage"))


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/24 06:21:51 WARN Utils: Your hostname, T14-PF4WM3XL, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/08/24 06:21:51 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


/home/angelalvarez/.local/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


26/08/24 06:21:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.2.0
WholeStageCodegen enabled: true


### Step 2: Build the controlled suspect
show(5) triggers the query so we know the execution path is real; this lab is not intended to force a full scan or benchmark performance.


In [2]:
q = (spark.range(0, 1_000_000).where((F.col("id") % 7) == 0).select((F.col("id") * 3 + 1).alias("x")))
q.show(5, truncate=False)


+---+
|x  |
+---+
|1  |
|22 |
|43 |
|64 |
|85 |
+---+
only showing top 5 rows


### Step 3: Inspect the ordinary physical plan
Look for repeated `*(1)` markers. Operators carrying the same marker are in the same WholeStageCodegen stage.


In [3]:
print("=== Ordinary physical plan ===")
q.explain()


=== Ordinary physical plan ===
== Physical Plan ==
*(1) Project [((id#0L * 3) + 1) AS x#4L]
+- *(1) Filter ((id#0L % 7) = 0)
   +- *(1) Range (0, 1000000, step=1, splits=2)




### Step 4: Inspect formatted codegen IDs
The formatted explanation exposes node details and the corresponding codegen IDs.


In [4]:
print("=== Formatted physical plan ===")
q.explain("formatted")


=== Formatted physical plan ===
== Physical Plan ==
* Project (3)
+- * Filter (2)
   +- * Range (1)


(1) Range [codegen id : 1]
Output [1]: [id#0L]
Arguments: Range (0, 1000000, step=1, splits=Some(2))

(2) Filter [codegen id : 1]
Input [1]: [id#0L]
Condition : ((id#0L % 7) = 0)

(3) Project [codegen id : 1]
Output [1]: [((id#0L * 3) + 1) AS x#4L]
Input [1]: [id#0L]




# 📊 Post-Lab Analysis: The WholeStageCodegen Stage Diagnostics

This lab forced a deliberately small `Range → Filter → Project` query through Spark SQL's physical planning and code-generation path.

The result is not a simple performance victory. It shows how to identify a generated execution stage, while keeping separate the question of **where codegen exists** from the question of **where a complete Spark job spends its time**.

### 1. One Pipeline, Three Operators

The ordinary physical plan assigns `*(1)` to `Project`, `Filter`, and `Range`. The formatted plan reports `codegen id : 1` for the same three operators.

That repeated marker is the key observation: Spark grouped the compatible operators into one WholeStageCodegen stage, allowing their generated execution to participate in the same pipeline.

### 2. The Formatted Plan Confirms the Identity

The normal plan gives us the readable `*(1)` clue. The formatted plan gives us the corresponding node-level evidence: `Range`, `Filter`, and `Project` each report the same codegen ID.

These are not two different plans. They are two views of the same physical decision.

### 3. What This Experiment Does Not Prove

This notebook does not prove that WholeStageCodegen makes the query faster. The workload contains no shuffle, storage wait, skew, spill, Python boundary, or expensive aggregation. There is no distributed bottleneck here for codegen to overcome.

It proves something narrower and more useful for the next investigation: **Spark generated execution for this physical-plan fragment, and we can point to the exact operators that share the stage.**

### 4. What to Carry into the Next Lab

When the plan becomes more complicated, look for changes in the evidence: different `*(n)` values, `Exchange` boundaries, `InputAdapter`, Python execution nodes, or operators that fall outside the generated island. Then compare the plan with runtime metrics before making a performance claim.

The star identifies the stage. The Spark UI explains the crime scene.
